In [1]:
import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd
import time

DATA_DIR = "../data"

# returns_and_market_cap contains monthly returns and market cap data per company
returns_and_market_cap = pd.read_csv(f"{DATA_DIR}/returns_market_cap.csv")

# ndt contains non-derivative transaction data (e.g., insider trades)
ndt = pd.read_csv(f"{DATA_DIR}/ndt.csv")

In [2]:
print("Non-derivative transaction data:")
print(ndt.head().to_markdown(index=False))
print()
print("Returns and market cap data:")
print(returns_and_market_cap.head().to_markdown(index=False))

Non-derivative transaction data:
| ACCESSION_NUMBER     | TRANS_DATE   | TRANS_CODE   |   EQUITY_SWAP_INVOLVED |   TRANS_SHARES |   TRANS_PRICEPERSHARE | TRANS_ACQUIRED_DISP_CD   |   SHRS_OWND_FOLWNG_TRANS | DIRECT_INDIRECT_OWNERSHIP   |   COMPANY_ID | RPTOWNERNAME     | RPTOWNER_RELATIONSHIP   | RPTOWNER_TITLE         |
|:---------------------|:-------------|:-------------|-----------------------:|---------------:|----------------------:|:-------------------------|-------------------------:|:----------------------------|-------------:|:-----------------|:------------------------|:-----------------------|
| 0000076605-17-000121 | 2017-09-27   | S            |                      0 |          10000 |                 83.16 | D                        |          15000           | D                           |       295249 | Cleveland Todd M | Director,Officer        | CEO                    |
| 0001140361-17-037031 | 2017-09-27   | F            |                      0 |          28833 | 

## 1: Data Preparation and Cleaning

In [3]:
# Convert date columns to datetime for consistent handling
# MONTH_END in returns_and_market_cap is the end-of-month date for returns/market cap
returns_and_market_cap['MONTH_END'] = pd.to_datetime(returns_and_market_cap['MONTH_END'])

# TRANS_DATE in ndt is the transaction date; use errors='coerce' to handle invalid dates
ndt['TRANS_DATE'] = pd.to_datetime(ndt['TRANS_DATE'], errors='coerce')

# Create a month-end column in ndt by rounding transaction dates to the nearest month end
ndt['month_end'] = ndt['TRANS_DATE'] + MonthEnd(0)

# Check for and drop rows with invalid transaction dates (NaT)
invalid_dates = ndt['TRANS_DATE'].isna().sum()
if invalid_dates > 0:
    print(f"Warning: Dropped {invalid_dates} rows with invalid TRANS_DATE.")
    ndt = ndt.dropna(subset=['TRANS_DATE'])

# Filter ndt to include only buy ('P') and sell ('S') transactions
trades = ndt[ndt['TRANS_CODE'].isin(['P', 'S'])].copy()

# ——————————————
# Collapse duplicate rows so each ACCESSION_NUMBER is one trade
# ——————————————
trades = trades.groupby('ACCESSION_NUMBER', as_index=False).agg({
    'TRANS_DATE': 'first',
    'TRANS_CODE': 'first',
    'EQUITY_SWAP_INVOLVED': 'first',
    'TRANS_SHARES': 'sum',
    'TRANS_PRICEPERSHARE': 'first',
    'TRANS_ACQUIRED_DISP_CD': 'first',
    'SHRS_OWND_FOLWNG_TRANS': 'first',
    'DIRECT_INDIRECT_OWNERSHIP': 'first',
    'COMPANY_ID': 'first',
    'RPTOWNERNAME': 'first',
    'RPTOWNER_RELATIONSHIP': 'first',
    'RPTOWNER_TITLE': 'first',
    'month_end': 'first'
})

# Calculate the dollar value of each trade (shares * price per share)
trades['trade_value'] = trades['TRANS_SHARES'] * trades['TRANS_PRICEPERSHARE']

print("Sample of trades data after preparation:")
print(trades.head().to_markdown(index=False))

Sample of trades data after preparation:
| ACCESSION_NUMBER     | TRANS_DATE          | TRANS_CODE   |   EQUITY_SWAP_INVOLVED |   TRANS_SHARES |   TRANS_PRICEPERSHARE | TRANS_ACQUIRED_DISP_CD   |   SHRS_OWND_FOLWNG_TRANS | DIRECT_INDIRECT_OWNERSHIP   |   COMPANY_ID | RPTOWNERNAME   | RPTOWNER_RELATIONSHIP   | RPTOWNER_TITLE                 | month_end           |      trade_value |
|:---------------------|:--------------------|:-------------|-----------------------:|---------------:|----------------------:|:-------------------------|-------------------------:|:----------------------------|-------------:|:---------------|:------------------------|:-------------------------------|:--------------------|-----------------:|
| 0000001750-06-000002 | 2006-01-04 00:00:00 | S            |                      0 |         144360 |                 24.64 | D                        |                  6876.17 | D                           |       168154 | STORCH DAVID P | Director,Officer        | C

## 2: Market Returns Calculation

In [4]:
# Compute total market capitalization per month for weighting returns
# Sum MARKET_CAP_USD across all companies for each MONTH_END
total_cap = returns_and_market_cap.groupby('MONTH_END')['MARKET_CAP_USD'].sum().rename('total_cap')

# Join total_cap back to returns_and_market_cap for weight calculation
returns_and_market_cap = returns_and_market_cap.join(total_cap, on='MONTH_END')

# Calculate each company's weight as its market cap divided by total market cap
returns_and_market_cap['weight'] = returns_and_market_cap['MARKET_CAP_USD'] / returns_and_market_cap['total_cap']

# Initialize a dictionary to store market returns for different horizons (1, 3, 12 months)
market_returns = {}

# Compute weighted market returns for each horizon
# For each month, multiply individual company returns by their weights and sum
for horizon in ['1', '3', '12']:
    market_returns[f'market_return_{horizon}m'] = returns_and_market_cap.groupby('MONTH_END').apply(
        lambda df: np.sum(df[f'RETURN_LEAD_{horizon}_MONTHS'] * df['weight']),
        include_groups=False  # Avoid including group keys in the apply function
    ).reset_index(name=f'market_return_{horizon}m')

for horizon in ['1', '3', '12']:
    print(f"Sample of market_return_{horizon}m market returns:")
    print(market_returns[f'market_return_{horizon}m'].head().to_markdown(index=False))

Sample of market_return_1m market returns:
| MONTH_END           |   market_return_1m |
|:--------------------|-------------------:|
| 1962-01-31 00:00:00 |          0.0886072 |
| 1962-02-28 00:00:00 |         -0.0697672 |
| 1962-03-31 00:00:00 |          0.0249994 |
| 1962-04-30 00:00:00 |         -0.414634  |
| 1962-05-31 00:00:00 |          0.0520832 |
Sample of market_return_3m market returns:
| MONTH_END           |   market_return_3m |
|:--------------------|-------------------:|
| 1962-01-31 00:00:00 |           0.037974 |
| 1962-02-28 00:00:00 |          -0.44186  |
| 1962-03-31 00:00:00 |          -0.36875  |
| 1962-04-30 00:00:00 |          -0.256097 |
| 1962-05-31 00:00:00 |           0.375    |
Sample of market_return_12m market returns:
| MONTH_END           |   market_return_12m |
|:--------------------|--------------------:|
| 1962-01-31 00:00:00 |           -0.319621 |
| 1962-02-28 00:00:00 |           -0.331395 |
| 1962-03-31 00:00:00 |           -0.234376 |
| 1962-04-

## 3: Merging Trades with Returns Data

In [5]:
# Merge trades with returns_and_market_cap to align trade data with returns and market cap
# Use left join to keep all trades, even if no matching returns data exists
merged = trades.merge(returns_and_market_cap, 
                      left_on=['COMPANY_ID', 'month_end'], 
                      right_on=['COMPANY_ID', 'MONTH_END'], 
                      how='left').drop(columns='MONTH_END')  # Drop redundant MONTH_END column

# Merge in the precomputed market returns for each horizon
for horizon, df in market_returns.items():
    merged = merged.merge(df, 
                          left_on='month_end', 
                          right_on='MONTH_END', 
                          how='left').drop(columns='MONTH_END')

# Calculate excess returns for each horizon (company return - market return)
for horizon in ['1', '3', '12']:
    merged[f'excess_return_{horizon}m'] = merged[f'RETURN_LEAD_{horizon}_MONTHS'] - merged[f'market_return_{horizon}m']

print("Sample of merged data with excess returns:")
print(merged[['COMPANY_ID', 'month_end', 'TRANS_CODE', 'excess_return_1m', 'excess_return_3m', 'excess_return_12m']].head().to_markdown(index=False))

Sample of merged data with excess returns:
|   COMPANY_ID | month_end           | TRANS_CODE   |   excess_return_1m |   excess_return_3m |   excess_return_12m |
|-------------:|:--------------------|:-------------|-------------------:|-------------------:|--------------------:|
|       168154 | 2006-01-31 00:00:00 | S            |          0.0584863 |          0.0745878 |           0.0870041 |
|       168154 | 2006-03-31 00:00:00 | S            |         -0.0752978 |         -0.193422  |          -0.165043  |
|       168154 | 2006-03-31 00:00:00 | S            |         -0.0752978 |         -0.193422  |          -0.165043  |
|       168154 | 2006-03-31 00:00:00 | S            |         -0.0752978 |         -0.193422  |          -0.165043  |
|       168154 | 2006-04-30 00:00:00 | S            |         -0.0544755 |         -0.0779137 |          -0.0112499 |


## 4: Net Trade Value and Quintile Assignment

In [6]:
# Start timing this section for performance tracking
start = time.perf_counter()

# Compute total trade value by company, month, and transaction type (P or S) in one pass
# Use unstack to pivot P and S into columns, filling missing values with 0
trade_values = trades.groupby(['COMPANY_ID', 'month_end', 'TRANS_CODE'])['trade_value'].sum().unstack(fill_value=0)

# Calculate net trade value as buys (P) minus sells (S)
# Use .get() to safely handle cases where 'P' or 'S' might be missing
net_value = (trade_values.get('P', 0) - trade_values.get('S', 0)).reset_index(name='net_trade_value')

# Merge net trade value back into the merged DataFrame
merged = merged.merge(net_value, on=['COMPANY_ID', 'month_end'], how='left')

# Define a helper function to assign quintiles robustly
# Forces 5 bins using qcut, falls back to equal-range bins if necessary
def safe_qcut(x, q=5):
    try:
        # Attempt to create 5 quantile-based bins
        return pd.qcut(x, q, labels=False, duplicates='drop') + 1
    except ValueError:
        # If qcut fails (e.g., too few unique values), use equal-range bins
        if x.notna().nunique() > 1:
            return pd.cut(x, bins=min(q, x.notna().nunique()), labels=False, include_lowest=True) + 1
        # If only one unique value or all NaN, assign to quintile 1
        return pd.Series(1, index=x.index)

# Assign quintiles based on net_trade_value within each month_end
merged['net_value_quintile'] = merged.groupby('month_end')['net_trade_value'].transform(safe_qcut)

print(f"Net trade value and quintile assignment elapsed: {time.perf_counter() - start:.4f} seconds")
print("Sample of merged data with net trade value and quintiles:")
print(merged[['COMPANY_ID', 'month_end', 'net_trade_value', 'net_value_quintile']].head().to_markdown(index=False))
print(
    "net_value_quintile legend:\n"
    "1 = lowest net trade value (biggest net sellers)\n"
    "2 = lower-middle net trade value\n"
    "3 = middle net trade value\n"
    "4 = upper-middle net trade value\n"
    "5 = highest net trade value (biggest net buyers)"
)

Net trade value and quintile assignment elapsed: 0.2148 seconds
Sample of merged data with net trade value and quintiles:
|   COMPANY_ID | month_end           |   net_trade_value |   net_value_quintile |
|-------------:|:--------------------|------------------:|---------------------:|
|       168154 | 2006-01-31 00:00:00 |      -3.55703e+06 |                    2 |
|       168154 | 2006-03-31 00:00:00 |      -1.12292e+07 |                    1 |
|       168154 | 2006-03-31 00:00:00 |      -1.12292e+07 |                    1 |
|       168154 | 2006-03-31 00:00:00 |      -1.12292e+07 |                    1 |
|       168154 | 2006-04-30 00:00:00 | -699651           |                    4 |
net_value_quintile legend:
1 = lowest net trade value (biggest net sellers)
2 = lower-middle net trade value
3 = middle net trade value
4 = upper-middle net trade value
5 = highest net trade value (biggest net buyers)


## 5: Performance Analysis of Net Trade Value

In [ ]:
# Filter to rows with at least one non-null excess return (more permissive than all three)
valid = merged.dropna(subset=[f'excess_return_{h}m' for h in ['1', '3', '12']], how='all').copy()
print(f"Dropped {len(merged) - len(valid)} rows due to missing excess returns.")

# Compute monthly average excess returns by quintile
monthly = valid.groupby(['month_end', 'net_value_quintile'])[[f'excess_return_{h}m' for h in ['1', '3', '12']]].mean().reset_index()

# Define a function to annualize returns
# Uses geometric compounding; falls back to arithmetic mean if cumulative return is non-positive
def annualize(series):
    series = series.dropna()  # Drop NaNs before calculation
    cum_return = (1 + series).prod()  # Cumulative return
    n = len(series)  # Number of periods
    # If positive cumulative return and data exists, annualize geometrically
    if cum_return > 0 and n > 0:
        return cum_return**(12/n) - 1
    # Otherwise, use arithmetic mean annualized (or NaN if no data)
    return series.mean() * 12 if n > 0 else np.nan

# Calculate annualized excess returns per quintile
agg = monthly.groupby('net_value_quintile').agg(**{
    f'ann_excess_{h}m': (f'excess_return_{h}m', annualize) for h in ['1', '3', '12']
}).reset_index()

# Count non-null monthly observations per quintile for each horizon
counts = monthly.groupby('net_value_quintile').agg(**{
    f'count_{h}m': (f'excess_return_{h}m', 'count') for h in ['1', '3', '12']
}).reset_index()

# Calculate hit rate: proportion of months where excess return > 0 (positive performance)
hit = valid.groupby('month_end').apply(
    lambda df: df.assign(**{f'beat_mean_{h}m': df[f'excess_return_{h}m'] > 0 for h in ['1', '3', '12']}),
    include_groups=False  # Exclude group keys from apply
).groupby('net_value_quintile')[[f'beat_mean_{h}m' for h in ['1', '3', '12']]].mean().reset_index().rename(
    columns=lambda c: f'hit_rate_{c[-2:]}' if 'beat' in c else c
)

# Combine all performance metrics into one DataFrame
performance = agg.merge(hit, on='net_value_quintile').merge(counts, on='net_value_quintile').sort_values('net_value_quintile')

print("Performance by net trade value quintile:")
print(performance.to_markdown())

print("\nNotes:")
print("- Annualized returns use geometric compounding; non-positive returns fall back to arithmetic mean * 12.")
print("- Hit rate measures the fraction of months where excess return > 0.")
print("- Counts show the avg number of monthly observations per quintile for each horizon.")

# when you have fat tailed returns the mean is shifted higher 
# when you have returns that aren't logged you will overestimate the return because you mean is dragged to the right
# compound over time go down 20 and back 20 to get the return 

Dropped 4011 rows due to missing excess returns.
Performance by net trade value quintile:
|    |   net_value_quintile |   ann_excess_1m |   ann_excess_3m |   ann_excess_12m |   hit_rate_1m |   hit_rate_3m |   hit_rate_2m |   count_1m |   count_3m |   count_12m |
|---:|---------------------:|----------------:|----------------:|-----------------:|--------------:|--------------:|--------------:|-----------:|-----------:|------------:|
|  0 |                    1 |     0.0279467   |      0.076064   |        0.0471924 |      0.50889  |      0.485463 |      0.475909 |        261 |        261 |         258 |
|  1 |                    2 |     0.000316425 |      0.00460364 |        0.178661  |      0.496354 |      0.490087 |      0.472416 |        238 |        238 |         235 |
|  2 |                    3 |     0.00956579  |      0.0752038  |        0.241705  |      0.499241 |      0.489959 |      0.461931 |        246 |        246 |         243 |
|  3 |                    4 |     0.0111522  

### Analyisis:
- Best short‑term performance (1–3m) comes from the largest net buyers (Quintile 1).
- Worst performance is clearly in Quintile 5 (biggest net sellers).
- Over 12 months, middle quintiles (2–3) still outperform extreme sellers.

In [8]:
from scipy.stats import ttest_1samp

# Filter valid observations
valid = valid.copy()

results = []
for quintile, df in valid.groupby('net_value_quintile'):
    for horizon in ['1', '3', '12']:
        series = df[f'excess_return_{horizon}m'].dropna()
        if len(series) < 5:
            # Skip tiny samples
            continue
        t_stat, p_value = ttest_1samp(series, 0)
        results.append({
            'net_value_quintile': quintile,
            'horizon_months': int(horizon),
            'mean_excess_return': series.mean(),
            't_stat': t_stat,
            'p_value': p_value,
            'n_obs': len(series)
        })

ttest_df = pd.DataFrame(results).sort_values(['net_value_quintile','horizon_months'])

print("One-sample t-test of excess returns vs zero by quintile:")
print(ttest_df.to_markdown(index=False))

One-sample t-test of excess returns vs zero by quintile:
|   net_value_quintile |   horizon_months |   mean_excess_return |   t_stat |     p_value |   n_obs |
|---------------------:|-----------------:|---------------------:|---------:|------------:|--------:|
|                    1 |                1 |           0.00190152 |  4.99213 | 5.98692e-07 |   65523 |
|                    1 |                3 |           0.0040022  |  5.54303 | 2.98423e-08 |   65523 |
|                    1 |               12 |           0.0130126  |  7.93179 | 2.19435e-15 |   64897 |
|                    2 |                1 |           0.00149497 |  3.66369 | 0.00024881  |   64458 |
|                    2 |                3 |           0.00324832 |  4.47404 | 7.68875e-06 |   64458 |
|                    2 |               12 |           0.0278175  | 15.0717  | 3.04705e-51 |   63835 |
|                    3 |                1 |           0.00384959 |  8.2852  | 1.20126e-16 |   65185 |
|                    3 | 

## Assign monthly size quintiles to market cap

In [9]:
# Ensure that the merged dataframe has a size_quintile assignment based on MARKET_CAP_USD for each month
merged['size_quintile'] = (
    merged
    .groupby('month_end')['MARKET_CAP_USD']
    .transform(lambda x: pd.qcut(x, 5, labels=False, duplicates='drop') + 1)
)

# (Optional) Recreate the valid dataset if needed so that it carries over the new column.
valid = merged.dropna(subset=[f'excess_return_{h}m' for h in ['1', '3', '12']], how='all').copy()

print("Sample of valid data with size_quintile assigned:")
print(valid[['COMPANY_ID', 'month_end', 'MARKET_CAP_USD', 'size_quintile']].head().to_markdown(index=False))
print(
    "size_quintile legend:\n"
    "1 = smallest 20% of companies by market cap\n"
    "2 = next 20%\n"
    "3 = middle 20%\n"
    "4 = next 20%\n"
    "5 = largest 20% of companies by market cap"
)

Sample of valid data with size_quintile assigned:
|   COMPANY_ID | month_end           |   MARKET_CAP_USD |   size_quintile |
|-------------:|:--------------------|-----------------:|----------------:|
|       168154 | 2006-01-31 00:00:00 |          798.269 |               2 |
|       168154 | 2006-03-31 00:00:00 |         1035.22  |               3 |
|       168154 | 2006-03-31 00:00:00 |         1035.22  |               3 |
|       168154 | 2006-03-31 00:00:00 |         1035.22  |               3 |
|       168154 | 2006-04-30 00:00:00 |          972.783 |               3 |
size_quintile legend:
1 = smallest 20% of companies by market cap
2 = next 20%
3 = middle 20%
4 = next 20%
5 = largest 20% of companies by market cap


## Compute performance by (size × net‑value) buckets

In [10]:
# 1️. Ensure size quintiles are assigned in the merged dataframe
merged['size_quintile'] = (
    merged
    .groupby('month_end')['MARKET_CAP_USD']
    .transform(lambda x: pd.qcut(x, 5, labels=False, duplicates='drop') + 1)
)

# Recreate the valid dataset so it carries the new 'size_quintile'
valid = merged.dropna(subset=[f'excess_return_{h}m' for h in ['1', '3', '12']], how='all').copy()

# 2️. Aggregate monthly excess returns by month, size_quintile, and net_value_quintile.
monthly_double = (
    valid.groupby(['month_end', 'size_quintile', 'net_value_quintile'])[
        [f'excess_return_{h}m' for h in ['1', '3', '12']]
    ]
    .mean()
    .reset_index()
)

# Function to annualize returns (geometric compounding or arithmetic fallback)
def annualize(series):
    series = series.dropna()  # Remove missing values
    cum_return = (1 + series).prod()  # Cumulative return
    n = len(series)  # Number of periods
    if cum_return > 0 and n > 0:
        return cum_return**(12/n) - 1  # Geometric annualization
    return series.mean() * 12 if n > 0 else np.nan  # Fallback: arithmetic

# Aggregate performance metrics for each double-sorted bucket
def agg_perf(df):
    return pd.Series({
        **{f'ann_excess_{h}m': annualize(df[f'excess_return_{h}m']) for h in ['1', '3', '12']},
        **{f'hit_rate_{h}m': (df[f'excess_return_{h}m'] > 0).mean() for h in ['1', '3', '12']},
        'n_obs': len(df)  # Number of monthly observations in this bucket
    })

# Group by size and net trade quintile, applying agg_perf while excluding grouping columns (fixes deprecation warning)
double_perf = (
    monthly_double
    .groupby(['size_quintile', 'net_value_quintile'])
    .apply(agg_perf, include_groups=False)
    .reset_index()
    .sort_values(['size_quintile', 'net_value_quintile'])
)

print("Double-sorted performance (by size quintile and net trade quintile):")
print(double_perf.to_markdown(index=False))

Double-sorted performance (by size quintile and net trade quintile):
|   size_quintile |   net_value_quintile |   ann_excess_1m |   ann_excess_3m |   ann_excess_12m |   hit_rate_1m |   hit_rate_3m |   hit_rate_12m |   n_obs |
|----------------:|---------------------:|----------------:|----------------:|-----------------:|--------------:|--------------:|---------------:|--------:|
|               1 |                    1 |      0.117247   |      -0.134982  |        0.795427  |      0.518717 |      0.518717 |       0.475936 |     187 |
|               1 |                    2 |     -0.0782273  |      -0.356713  |       -0.632581  |      0.452055 |      0.406393 |       0.39726  |     219 |
|               1 |                    3 |     -0.0727928  |      -0.177912  |       -0.145444  |      0.443478 |      0.469565 |       0.46087  |     230 |
|               1 |                    4 |      0.0757412  |       0.0608577 |        0.919624  |      0.523605 |      0.48927  |       0.579399 |

- Hit rate ~50% across almost every bucket means none consistently beat the market far more often than they lose. 
- Small stocks that sold the most (size_quintile 1 & net_value_quintile 1) had the strongest short‑term (1‑month) and especially long‑term (12‑month) excess returns.
- Big buyers (net_value_quintile 5) tended to underperform the market, especially in small firms.
- Some medium‑large buckets (size 5, net_value_quintile 2–3) show modest positive returns and hit rates above 55% over 12 months.

##### Overall, selling activity in small companies appears linked to higher future returns, but the pattern weakens for larger firms and isn’t ironclad (hit rates hover around 50%).

## pre‑2015 vs post‑2015 Does the “buyer premium” hold throughout?

In [11]:
# Add a new column "period" to indicate pre‑2015 vs post‑2015
valid['period'] = np.where(valid['month_end'] < pd.Timestamp('2015-01-01'), 'Pre-2015', 'Post-2015')

# Check how many observations are in each period
print("Observations by period:")
print(valid['period'].value_counts())

Observations by period:
period
Pre-2015     176575
Post-2015    146048
Name: count, dtype: int64


## Compute Performance Metrics by Period and Net Trade Quintile

In [12]:
# Aggregate monthly average excess returns by month, period, and net trade quintile
monthly_period = valid.groupby(['month_end', 'period', 'net_value_quintile'])[
    [f'excess_return_{h}m' for h in ['1', '3', '12']]
].mean().reset_index()

# Use the same annualization function as before
def annualize(series):
    series = series.dropna()  # Remove missing values
    cum_return = (1 + series).prod()  # Cumulative return
    n = len(series)
    if cum_return > 0 and n > 0:
        return cum_return**(12/n) - 1
    return series.mean() * 12 if n > 0 else np.nan

# Define an aggregation function for performance metrics
def agg_perf(df):
    return pd.Series({
        **{f'ann_excess_{h}m': annualize(df[f'excess_return_{h}m']) for h in ['1', '3', '12']},
        **{f'hit_rate_{h}m': (df[f'excess_return_{h}m'] > 0).mean() for h in ['1', '3', '12']},
        'n_obs': len(df)
    })

# Compute performance metrics for each period and net trade quintile
performance_period = (
    monthly_period
    .groupby(['period', 'net_value_quintile'])
    .apply(agg_perf, include_groups=False)
    .reset_index()
    .sort_values(['period', 'net_value_quintile'])
)

print("Performance by net trade quintile and period:")
print(performance_period.to_markdown(index=False))

Performance by net trade quintile and period:
| period    |   net_value_quintile |   ann_excess_1m |   ann_excess_3m |   ann_excess_12m |   hit_rate_1m |   hit_rate_3m |   hit_rate_12m |   n_obs |
|:----------|---------------------:|----------------:|----------------:|-----------------:|--------------:|--------------:|---------------:|--------:|
| Post-2015 |                    1 |      0.0600724  |      0.101568   |        0.206959  |      0.535088 |      0.552632 |       0.54386  |     114 |
| Post-2015 |                    2 |      0.0176829  |      0.0348064  |        0.117983  |      0.508772 |      0.570175 |       0.5      |     114 |
| Post-2015 |                    3 |      0.0161948  |      0.00344769 |        0.0398134 |      0.482456 |      0.473684 |       0.5      |     114 |
| Post-2015 |                    4 |      0.029157   |     -0.0380641  |       -0.0119484 |      0.495575 |      0.442478 |       0.460177 |     113 |
| Post-2015 |                    5 |     -0.0515

## Run T‑Tests for Excess Returns by Period and Quintile

In [13]:
from scipy.stats import ttest_1samp

results_period = []
# Group first by period, then by net trade quintile
for period, period_df in valid.groupby('period'):
    for quintile, df in period_df.groupby('net_value_quintile'):
        for horizon in ['1', '3', '12']:
            series = df[f'excess_return_{horizon}m'].dropna()
            if len(series) < 5:  # Skip tiny samples
                continue
            t_stat, p_value = ttest_1samp(series, 0)
            results_period.append({
                'period': period,
                'net_value_quintile': quintile,
                'horizon_months': int(horizon),
                'mean_excess_return': series.mean(),
                't_stat': t_stat,
                'p_value': p_value,
                'n_obs': len(series)
            })

ttest_df_period = pd.DataFrame(results_period).sort_values(['period', 'net_value_quintile', 'horizon_months'])
print("One-sample t-test of excess returns vs zero by period and quintile:")
print(ttest_df_period.to_markdown(index=False))

One-sample t-test of excess returns vs zero by period and quintile:
| period    |   net_value_quintile |   horizon_months |   mean_excess_return |    t_stat |     p_value |   n_obs |
|:----------|---------------------:|-----------------:|---------------------:|----------:|------------:|--------:|
| Post-2015 |                    1 |                1 |          0.00392502  |  6.34858  | 2.20457e-10 |   29668 |
| Post-2015 |                    1 |                3 |          0.0103618   |  9.14256  | 6.47716e-20 |   29668 |
| Post-2015 |                    1 |               12 |          0.0219208   |  7.90719  | 2.72554e-15 |   29042 |
| Post-2015 |                    2 |                1 |          0.000618564 |  0.999831 | 0.317401    |   29178 |
| Post-2015 |                    2 |                3 |          0.00356962  |  3.16493  | 0.0015528   |   29178 |
| Post-2015 |                    2 |               12 |          0.019111    |  6.36643  | 1.96416e-10 |   28555 |
| Post-2015 